In [1]:
#Data Visualization
#1.prepare data for visualization
#2.anlyiz number of rides by
   #season
   #month
   #compare with weather conditions
#3.Visualize Top Start Stations
#4.Visualize Top End Stations
#5.Homework Part 2

In [2]:
import requests

import numpy as np
import pandas as pd

import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

In [3]:
weather_daily = pd.read_csv('../data/citibike/JC/jersey_weather_2025.csv')
citibike_df = pd.read_csv('../data/citibike/JC/JC2025_Enriched.csv')

In [4]:
citibike_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1998450 entries, 0 to 1998449
Data columns (total 20 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   ride_id                str    
 1   rideable_type          str    
 2   started_at             str    
 3   ended_at               str    
 4   start_station_name     str    
 5   start_station_id       str    
 6   end_station_name       str    
 7   end_station_id         str    
 8   start_lat              float64
 9   start_lng              float64
 10  end_lat                float64
 11  end_lng                float64
 12  member_casual          str    
 13  ride_duration_minutes  float64
 14  date                   str    
 15  month                  str    
 16  month_name             str    
 17  day_of_week            str    
 18  hour                   int64  
 19  season                 str    
dtypes: float64(5), int64(1), str(14)
memory usage: 304.9 MB


In [5]:
citibike_df['date'] = pd.to_datetime(citibike_df['date'], errors='coerce')

In [6]:
weather_daily.info()

<class 'pandas.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   temperature_2m_max   365 non-null    float64
 1   temperature_2m_min   365 non-null    float64
 2   temperature_2m_mean  365 non-null    float64
 3   precipitation_sum    365 non-null    float64
 4   rain_sum             365 non-null    float64
 5   snowfall_sum         365 non-null    float64
 6   wind_speed_10m_max   365 non-null    float64
 7   date                 365 non-null    str    
dtypes: float64(7), str(1)
memory usage: 22.9 KB


#### Number of rides per monhts

In [7]:
#First, we create a monthly aggregation.
#Each row will represent one month.
monthly_rides = (
    citibike_df
    .groupby("month", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

monthly_rides

,month,number_of_rides
0,2024-12,4
1,2025-01,101178
2,2025-02,90500
3,2025-03,146554
4,2025-04,163066
5,2025-05,186404
6,2025-06,193472
7,2025-07,214748
8,2025-08,216002
9,2025-09,231160


In [8]:
# Visualization the number of rides per month.

fig = px.bar(
    monthly_rides,
    x="month",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Month"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Rides",
)

fig.show()

#### Number of Rides per Season


In [9]:
#Now we aggregate the data by season.
season_rides = (
    citibike_df
    .groupby("season", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

season_order = ["Winter", "Spring", "Summer", "Autumn"]

season_rides["season"] = pd.Categorical(
    season_rides["season"],
    categories=season_order,
    ordered=True
)

season_rides = season_rides.sort_values("season")

season_rides

,season,number_of_rides
3,Winter,287406
1,Spring,496024
2,Summer,624222
0,Autumn,590798


In [10]:
#Vizualization by Season
fig = px.bar(
    season_rides,
    x="season",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Season",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Season",
    yaxis_title="Number of Rides"
)

fig.show()

#### Top 10 Start Stations


In [11]:
#First, we calculate the top 10 stations by number of departures.
top_start_stations = (
    citibike_df
    .dropna(subset=["start_station_name"])
    .groupby("start_station_name", as_index=False)
    .agg(
        number_of_departures=("ride_id", "count")
    )
    .sort_values("number_of_departures", ascending=False)
    .head(10)
)

top_start_stations


,start_station_name,number_of_departures
52,Grove St PATH,90008
58,Hoboken Terminal - Hudson St & Hudson Pl,51778
53,Hamilton Park,44518
95,River St & Newark St,42766
86,Newport PATH,41326
18,Bergen Ave & Sip Ave,40796
44,Exchange Pl,40016
0,11 St & Washington St,38962
94,River St & 1 St,38286
87,Newport Pkwy,37440


In [12]:
fig = px.bar(
    top_start_stations.sort_values("number_of_departures"),
    x="number_of_departures",
    y="start_station_name",
    orientation="h",
    title="Top 10 Start Stations by Number of Departures",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Departures",
    yaxis_title="Start Station"
)

fig.show()

#### Top 10 End Stations


In [13]:
#Now we calculate the top 10 stations by number of arrivals.
top_end_stations = (
    citibike_df
    .dropna(subset=["end_station_name"])
    .groupby("end_station_name", as_index=False)
    .agg(
        number_of_arrivals=("ride_id", "count")
    )
    .sort_values("number_of_arrivals", ascending=False)
    .head(10)
)

top_end_stations

,end_station_name,number_of_arrivals
232,Grove St PATH,95516
241,Hoboken Terminal - Hudson St & Hudson Pl,53278
233,Hamilton Park,44706
347,River St & Newark St,44226
317,Newport PATH,41402
73,Bergen Ave & Sip Ave,40736
207,Exchange Pl,40320
7,11 St & Washington St,39010
318,Newport Pkwy,37414
346,River St & 1 St,37030


In [14]:
fig = px.bar(
    top_end_stations.sort_values("number_of_arrivals"),
    x="number_of_arrivals",
    y="end_station_name",
    orientation="h",
    title="Top 10 End Stations by Number of Arrivals",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Arrivals",
    yaxis_title="End Station"
)

fig.show()

#### Merge Citibike with Daily Rides

In [15]:
daily_rides = (
    citibike_df
    .groupby("date", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)
daily_rides["date"] = pd.to_datetime(daily_rides["date"])
daily_rides.head()

,date,number_of_rides
0,2024-12-31,4
1,2025-01-01,2358
2,2025-01-02,3420
3,2025-01-03,3540
4,2025-01-04,2674


In [16]:
weather_daily.head()

,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max,date
0,11.0,3.9,7.4,4.5,4.5,0.0,23.2,2025-01-01
1,5.4,0.3,2.6,0.0,0.0,0.0,25.1,2025-01-02
2,3.2,-1.9,0.4,0.0,0.0,0.0,17.1,2025-01-03
3,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1,2025-01-04
4,0.4,-3.6,-2.2,0.0,0.0,0.0,19.9,2025-01-05


In [17]:
weather_daily["date"] = pd.to_datetime(weather_daily["date"])

weather_daily.dtypes

temperature_2m_max            float64
temperature_2m_min            float64
temperature_2m_mean           float64
precipitation_sum             float64
rain_sum                      float64
snowfall_sum                  float64
wind_speed_10m_max            float64
date                   datetime64[us]
dtype: object

In [18]:
#Now we merge the two datasets.
bike_weather_daily = daily_rides.merge(
    weather_daily,
    on="date",
    how="left"
)

bike_weather_daily.head()

,date,number_of_rides,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2024-12-31,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01,2358,11.0,3.9,7.4,4.5,4.5,0.0,23.2
2,2025-01-02,3420,5.4,0.3,2.6,0.0,0.0,0.0,25.1
3,2025-01-03,3540,3.2,-1.9,0.4,0.0,0.0,0.0,17.1
4,2025-01-04,2674,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1


In [19]:
#Check missing values after the merge.
bike_weather_daily.isna().sum()

date                   0
number_of_rides        0
temperature_2m_max     1
temperature_2m_min     1
temperature_2m_mean    1
precipitation_sum      1
rain_sum               1
snowfall_sum           1
wind_speed_10m_max     1
dtype: int64

#### Rides and Average Temperature


In [21]:
import plotly.express as px
import statsmodels.api as sm

In [22]:
fig = px.scatter(
    bike_weather_daily,
    x="temperature_2m_mean",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Average Temperature"
)

fig.update_layout(
    xaxis_title="Average Daily Temperature",
    yaxis_title="Number of Rides"
)

fig.show()

#### Rides and Wind Speed


In [23]:
fig = px.scatter(
    bike_weather_daily,
    x="wind_speed_10m_max",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Maximum Wind Speed"
)

fig.update_layout(
    xaxis_title="Maximum Wind Speed",
    yaxis_title="Number of Rides"
)

fig.show()

#### Rides and Precipitation


In [24]:
fig = px.scatter(
    bike_weather_daily,
    x="precipitation_sum",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Precipitation"
)

fig.update_layout(
    xaxis_title="Daily Precipitation",
    yaxis_title="Number of Rides"
)

fig.show()

#### Dual y axis | rides vs temperature

In [25]:
import plotly.graph_objects as go


fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=bike_weather_daily["date"],
        y=bike_weather_daily["number_of_rides"],
        mode="lines",
        name="Daily Rides",
        yaxis="y1"
    )
)

fig.add_trace(
    go.Scatter(
        x=bike_weather_daily["date"],
        y=bike_weather_daily["temperature_2m_mean"],
        mode="lines",
        name="Daily Average Temperature",
        yaxis="y2"
    )
)

fig.update_layout(
    title="Daily Rides and Daily Average Temperature",
    xaxis=dict(
        title="Day"
    ),
    yaxis=dict(
        title="Daily Rides",
        side="left"
    ),
    yaxis2=dict(
        title="Daily Average Temperature",
        overlaying="y",
        side="right"
    ),
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()